# 🎓 LoRA Fine-tuning v2 (Tutorial-based / KcBERT)

**보정된 UnSmile + 수집된 게임 음성채팅 데이터**로 학습합니다.

- **모델**: `beomi/kcbert-base`
- **메트릭**: `LRAP`
- **데이터**: UnSmile 보정 10,490 + 수집 519 = **11,009건**

In [1]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA L40S


In [2]:
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./output/lora_tutorial_kcbert_v2"  # v2 폴더 내 output 폴더에 저장
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS, BATCH_SIZE, LEARNING_RATE = 5, 64, 2e-4
MAX_LENGTH = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
unsmile_train = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
collected_df = pd.read_csv("../../1_Data_Labeling_STT/keywords_unsmile_format.tsv", sep='\t')

train_df = pd.concat([unsmile_train, collected_df], ignore_index=True)
print(f"✅ Train: {len(train_df)}건, Valid: {len(valid_df)}건")

✅ Train: 15208건, Valid: 3663건


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

Map:   0%|          | 0/15208 [00:00<?, ? examples/s]

Map:   0%|          | 0/3663 [00:00<?, ? examples/s]

In [5]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification")
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=["query", "key", "value"], bias="none")
model = get_peft_model(base_model, peft_config).to(DEVICE)
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 892,426 || all params: 109,818,644 || trainable%: 0.8126


In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    return {'lrap': label_ranking_average_precision_score(labels, predictions)}

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="lrap", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer=tokenizer))

In [8]:
print("🚀 v2 학습 시작...")
trainer.train()
print("학습 완료!")

🚀 v2 학습 시작...


Epoch,Training Loss,Validation Loss,Lrap
1,0.385000,0.287046,0.560413
2,0.279500,0.248392,0.711045
3,0.250300,0.208397,0.767799
4,0.218700,0.190677,0.799185
5,0.192100,0.185635,0.802053


학습 완료!


In [9]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
print("✅ v2 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")

✅ v2 모델 저장 완료!


  eval_loss: 0.1856
  eval_lrap: 0.8021
  eval_runtime: 2.1234
  eval_samples_per_second: 1725.0550
  eval_steps_per_second: 7.0640
  epoch: 5.0000
